# 1. Anonimización de Datos — Ataque de Enlace (Linkage Attack)

---

## 0. Carga del dataset Adult Census Income

In [1]:
import pandas as pd
import numpy as np

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, names=COLUMNS, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Dataset cargado: {len(df)} registros, {df.shape[1]} atributos')
df.head()

Dataset cargado: 32561 registros, 15 atributos


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 1. Clasificación de atributos

### Identificadores directos
El dataset Adult Census Income **no contiene identificadores directos** (nombre, DNI, email, teléfono, pasaporte). Esta es una característica habitual de datasets publicados para investigación.

### Cuasi-identificadores (QI)
Son atributos que, combinados, pueden reidentificar a un individuo al cruzarse con fuentes externas:

| Atributo | Estrategias de anonimización |
|---|---|
| `age` | Generalización por rangos (20-30, 30-40...) · Supresión de edades extremas |
| `sex` | Supresión del atributo · Generalización a categoría única |
| `native_country` | Generalización geográfica (región/continente) · Supresión de países minoritarios |
| `race` | Supresión del atributo · Generalización a categoría 'Persona' |
| `education` | Generalización por niveles (básico/medio/superior) |
| `occupation` | Generalización por sectores |

### Atributos sensibles
Información privada cuya divulgación supone el riesgo principal:
- `income` (categoría de salario: **atributo sensible principal**)
- `capital_gain`, `capital_loss`
- `occupation`

### Atributos de baja sensibilidad / estadísticos
- `fnlwgt`, `education_num`, `marital_status`, `relationship`, `hours_per_week`

## 2. Implementación de k-anonimato (k=3)

### Técnicas aplicadas por atributo

| Atributo | Técnica | Justificación |
|---|---|---|
| `age` | **Generalización** por rangos de 10 años | La edad exacta es muy identificativa. Agrupar en bandas (30-39) reduce granularidad sin perder utilidad analítica. |
| `sex` | Sin modificación | Solo 2 valores; la generalización eliminaría toda la información. Se controla mediante supresión de grupos pequeños. |
| `native_country` | **Supresión** de países con <100 registros → `'Other'` | Muchos países tienen pocos representantes, lo que genera grupos únicos por sí solos. |

In [2]:
def anonimizar_age(age: int) -> str:
    """Generalización por rangos de 10 años."""
    low = (int(age) // 10) * 10
    return f'{low}-{low + 9}'


def anonimizar_country(series: pd.Series, min_freq: int = 100) -> pd.Series:
    """Supresión: países con < min_freq registros → 'Other'."""
    counts = series.value_counts()
    rare = counts[counts < min_freq].index
    return series.apply(lambda x: 'Other' if x in rare else x)


def aplicar_k_anonimato(df: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Aplica k-anonimato sobre age, sex y native_country.
    Garantiza que cada combinación de QI aparece al menos k veces.
    """
    df_anon = df.copy()
    # Paso 1: Generalizar age
    df_anon['age'] = df_anon['age'].apply(anonimizar_age)
    # Paso 2: Suprimir países minoritarios
    df_anon['native_country'] = anonimizar_country(df_anon['native_country'], min_freq=100)
    # Paso 3: Eliminar registros cuyo grupo QI tiene menos de k individuos
    qi = ['age', 'sex', 'native_country']
    group_sizes = df_anon.groupby(qi)[qi[0]].transform('count')
    df_anon = df_anon[group_sizes >= k].copy()
    return df_anon


df_anon = aplicar_k_anonimato(df, k=3)

qi = ['age', 'sex', 'native_country']
min_group = df_anon.groupby(qi).size().min()

print(f'Registros originales  : {len(df)}')
print(f'Registros anonimizados: {len(df_anon)}')
print(f'Tamaño mínimo de grupo QI: {min_group}  (debe ser ≥ 3)')
print(f'k-anonimato garantizado: {"SÍ ✓" if min_group >= 3 else "NO ✗"}')
df_anon[['age', 'sex', 'native_country', 'income']].head(10)

Registros originales  : 32561
Registros anonimizados: 32519
Tamaño mínimo de grupo QI: 3  (debe ser ≥ 3)
k-anonimato garantizado: SÍ ✓


,age,sex,native_country,income
0,30-39,Male,United-States,<=50K
1,50-59,Male,United-States,<=50K
2,30-39,Male,United-States,<=50K
3,50-59,Male,United-States,<=50K
4,20-29,Female,Other,<=50K
5,30-39,Female,United-States,<=50K
6,40-49,Female,Other,<=50K
7,50-59,Male,United-States,>50K
8,30-39,Female,United-States,>50K
9,40-49,Male,United-States,>50K


## 3. ¿Es suficiente anonimizar solo un atributo?

Comprobaremos que anonimizar únicamente `age` o únicamente `native_country` es **insuficiente** para garantizar k-anonimato.

In [3]:
def verificar_k(df_test, qi, k=3, label=''):
    sizes = df_test.groupby(qi).size()
    violaciones = (sizes < k).sum()
    print(f'[{label}]')
    print(f'  Grupos QI totales             : {len(sizes)}')
    print(f'  Grupos con tamaño < {k} (violac.) : {violaciones}')
    print(f'  k-anonimato garantizado        : {"SÍ ✓" if violaciones == 0 else "NO ✗"}')
    if violaciones > 0:
        print('  Ejemplos de grupos violadores:')
        display(sizes[sizes < k].head(5).to_frame('count').reset_index())
    print()


# Solo age generalizado
df_solo_age = df.copy()
df_solo_age['age'] = df_solo_age['age'].apply(anonimizar_age)
verificar_k(df_solo_age, qi=['age', 'sex', 'native_country'],
            label='Solo age generalizado')

# Solo native_country suprimido
df_solo_country = df.copy()
df_solo_country['native_country'] = anonimizar_country(df_solo_country['native_country'])
verificar_k(df_solo_country, qi=['age', 'sex', 'native_country'],
            label='Solo native_country suprimido')

[Solo age generalizado]
  Grupos QI totales             : 449
  Grupos con tamaño < 3 (violac.) : 180
  k-anonimato garantizado        : NO ✗
  Ejemplos de grupos violadores:


,age,sex,native_country,count
0,10-19,Female,Cambodia,1
1,10-19,Female,Dominican-Republic,1
2,10-19,Female,El-Salvador,2
3,10-19,Female,Germany,1
4,10-19,Female,Guatemala,1



[Solo native_country suprimido]
  Grupos QI totales             : 874
  Grupos con tamaño < 3 (violac.) : 412
  k-anonimato garantizado        : NO ✗
  Ejemplos de grupos violadores:


,age,sex,native_country,count
0,17,Female,?,1
1,17,Female,India,1
2,17,Female,Mexico,2
3,17,Female,Other,2
4,17,Female,Philippines,1


### Conclusión
Al mantener la precisión de los demás cuasi-identificadores, siguen existiendo combinaciones que identifican a un único individuo (grupos de tamaño 1 o 2). Un atacante solo necesita conocer los atributos no anonimizados para cruzarlos con una fuente externa y reidentificar al sujeto. **Es imprescindible anonimizar conjuntamente todos los cuasi-identificadores.**

## 4. Ataque de enlace (Linkage Attack)

### Explicación teórica

Un atacante dispone de:
- **A)** El dataset anonimizado (age en rangos, native_country suprimido, sin identificadores directos).
- **B)** Un dataset externo con nombre, pasaporte, age exacto, sex y native_country obtenidos de fuentes públicas (redes sociales, registros electorales, bases de datos comerciales, etc.).

**Proceso del ataque:**
1. El atacante aplica las mismas transformaciones de generalización al dataset externo (age → rango, países minoritarios → 'Other').
2. Cruza (JOIN) el dataset externo con el anonimizado usando los cuasi-identificadores como clave de cruce.
3. Cuando el cruce devuelve un único valor de `income` para un individuo, la reidentificación es inequívoca.

**¿Por qué falla k-anonimato aquí?**  
k-anonimato garantiza grupos de tamaño ≥ k *dentro del dataset anonimizado*, pero no tiene en cuenta la distribución del espacio de QI en el dataset externo. Si el dataset externo contiene exactamente 1 individuo que coincide con un grupo, ese individuo queda reidentificado sin importar el tamaño del grupo.

In [5]:
# Cargar dataset externo ficticio
ext = pd.read_csv('external_dataset.csv')
print('Dataset externo:')
display(ext)

# Aplicar las MISMAS transformaciones de anonimización al dataset externo
countries_kept = set(df_anon['native_country'].unique())
ext['age_range'] = ext['age'].apply(anonimizar_age)
ext['native_country_anon'] = ext['native_country'].apply(
    lambda x: x if x in countries_kept else 'Other'
)

print('\nDataset externo con QI anonimizados:')
display(ext[['fictitious_name', 'passport_id', 'age_range', 'sex', 'native_country_anon']])

Dataset externo:


,fictitious_name,passport_id,hours_per_week,education,native_country,sex,age,employer_or_position
0,Alex Carter,P730000,3,Bachelors,Hungary,Female,81,Walmart
1,Jordan Diaz,P730001,40,Bachelors,England,Female,90,Amazon
2,Taylor Foster,P730002,50,HS-grad,Ecuador,Male,90,Target
3,Casey Patel,P730003,15,1st-4th,Poland,Male,81,Costco
4,Morgan Reed,P730004,24,HS-grad,United-States,Female,90,The Home Depot
5,Riley Hayes,P730005,40,5th-6th,NaN,Female,70,US Postal Service Clerk
6,Avery Kim,P730006,28,10th,United-States,Female,82,State Government Program Analyst
7,Parker Brooks,P730007,41,Some-college,Vietnam,Male,71,City Administrative Assistant
8,Quinn Bennett,P730008,40,HS-grad,United-States,Male,90,IRS Revenue Agent
9,Skyler Morris,P730009,16,11th,South,Female,20,County Public Works Coordinator



Dataset externo con QI anonimizados:


,fictitious_name,passport_id,age_range,sex,native_country_anon
0,Alex Carter,P730000,80-89,Female,Other
1,Jordan Diaz,P730001,90-99,Female,Other
2,Taylor Foster,P730002,90-99,Male,Other
3,Casey Patel,P730003,80-89,Male,Other
4,Morgan Reed,P730004,90-99,Female,United-States
5,Riley Hayes,P730005,70-79,Female,Other
6,Avery Kim,P730006,80-89,Female,United-States
7,Parker Brooks,P730007,70-79,Male,Other
8,Quinn Bennett,P730008,90-99,Male,United-States
9,Skyler Morris,P730009,20-29,Female,Other


In [8]:
# Preparar dataset anonimizado para el cruce
df_anon_merge = df_anon[['age', 'sex', 'native_country', 'income']].copy()
df_anon_merge.columns = ['age_range', 'sex', 'native_country_anon', 'income']

# JOIN por cuasi-identificadores
merged = ext.merge(
    df_anon_merge,
    on=['age_range', 'sex', 'native_country_anon'],
    how='inner'
)

# Solo reidentificaciones inequívocas (el income es único en todos los matches)
reidentificados = (
    merged.groupby(['fictitious_name', 'passport_id'])
    .filter(lambda g: g['income'].nunique() == 1)
    .drop_duplicates(subset=['fictitious_name', 'passport_id'])
)

print('REGISTROS REIDENTIFICADOS INEQUÍVOCAMENTE:')
display(reidentificados[['fictitious_name', 'passport_id', 'age_range', 'sex', 'native_country_anon', 'income']])
print(f'\nTotal reidentificados: {len(reidentificados)} de {len(ext)} individuos en el dataset externo')

REGISTROS REIDENTIFICADOS INEQUÍVOCAMENTE:


,fictitious_name,passport_id,age_range,sex,native_country_anon,income
0,Casey Patel,P730003,80-89,Male,Other,<=50K
15,Riley Hayes,P730005,70-79,Female,Other,<=50K
21,Avery Kim,P730006,80-89,Female,United-States,<=50K



Total reidentificados: 3 de 10 individuos en el dataset externo


## 5. ¿Habría evitado el ataque l-diversity o t-closeness?

### l-diversity
l-diversity exige que dentro de cada grupo de k-anonimato existan al menos **l valores distintos** del atributo sensible (`income`). En nuestro caso, cada combinación `(age_range, sex, native_country)` debería contener registros de **ambas** categorías de income (`<=50K` y `>50K`).

**¿Habría evitado el ataque?** **Parcialmente.** l-diversity protege contra ataques de homogeneidad (cuando todos en el grupo tienen el mismo income), pero **no evita el linkage attack** cuando el cruce de cuasi-identificadores produce grupos pequeños en el espacio del atacante. Si el atacante tiene información de contexto adicional (sector de trabajo, nivel educativo, etc.), puede seguir inferiendo el income.

### t-closeness
t-closeness exige que la distribución del atributo sensible dentro de cada grupo no se aleje más de una distancia **t** de la distribución global. Esto previene inferencias estadísticas sobre el atributo sensible dentro de un grupo.

**¿Habría evitado el ataque?** **Tampoco completamente.** t-closeness **no evita la reidentificación de la identidad** del individuo mediante el cruce con datos externos. Reduce la información que el atacante obtiene sobre `income` una vez reidentificado, pero si el grupo tiene exactamente k=3 registros y el atacante localiza al individuo en ese grupo, puede conocer su income con probabilidad 1/3 en el peor caso.

### Conclusión
Ninguna de las tres técnicas protege completamente frente a un ataque de enlace cuando el atacante dispone de un dataset externo con cuasi-identificadores precisos. Las soluciones más robustas son:
1. **Privacidad diferencial**: añadir ruido calibrado matemáticamente para garantizar protección formal.
2. Reducir la granularidad publicada de los cuasi-identificadores.
3. Controlar el acceso al dataset (no publicarlo abiertamente).
4. Aplicar supresión más agresiva en grupos pequeños.